# ch12. 그래디언트와 경사하강 — AI 가 '배우는' 원리 🎓 (마지막)

웹 챕터 12에서 배운 내용을 직접 구현합니다.

**이 노트북에서 하는 것**
1. **경사하강법을 직접 구현** — 공이 골짜기로 내려간 자취 그리기 (시작점 따라 다른 골짜기!)
2. **학습률 실험** — 너무 작으면 느림 / 적당 / 너무 크면 발산
3. **수치미분으로 기울기 검산** — 기울기 공식이 맞는지 컴퓨터로 확인
4. **벡터 버전(2변수)** — 등고선 위에서 ∇L 반대 방향으로 내려가는 경로

> 용어 복습 — **손실 L(w)**: 틀린 정도. **기울기 L'(w)**: 발밑 경사(양수=오르막).
> **경사하강**: $w \leftarrow w - \eta L'(w)$ — 여기서 ← 는 **대입 화살표**(왼쪽 값을 오른쪽
> 계산 결과로 새로 바꿔 넣기)입니다. **학습률 η**: 한 걸음 크기.
> **그래디언트 ∇L**: 기울기들을 모은 벡터 — 가장 가파른 오르막 방향(그래서 반대로 간다).

> 🤖 **AI 튜터** — 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요 — 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch12')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정 (그래프의 한글이 깨지지 않게)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print('numpy', np.__version__, '— 준비 완료')

## 1. 경사하강법 직접 구현

웹 데모와 같은, **골짜기가 두 개**인 손실 곡선을 씁니다.

$$L(w) = \frac{(w^2-4)^2}{8} + 0.3w + 1, \qquad L'(w) = \frac{w(w^2-4)}{2} + 0.3$$

구현은 딱 세 줄 — "기울기 계산 → 반대로 한 걸음 → 반복".

In [ ]:
def L(w):
    return (w**2 - 4) ** 2 / 8 + 0.3 * w + 1

def dL(w):
    return w * (w**2 - 4) / 2 + 0.3

def gradient_descent(w0, lr, n_steps=60):
    """시작점 w0 에서 경사하강. 지나온 자취(w 들)를 돌려준다."""
    path = [w0]
    w = w0
    for _ in range(n_steps):
        w = w - lr * dL(w)          # 핵심 한 줄: 기울기 반대 방향으로 한 걸음
        path.append(w)
        if abs(w) > 1e3:            # 발산하면 중단
            break
    return np.array(path)

path_R = gradient_descent(w0=3.2, lr=0.1)    # 오른쪽에서 출발
path_C = gradient_descent(w0=-0.6, lr=0.1)   # 가운데서 출발

ws = np.linspace(-3.6, 3.6, 300)
plt.figure(figsize=(7, 4.5))
plt.plot(ws, L(ws), color='#6366f1', lw=2, label='손실 곡선 L(w)')
plt.plot(path_R, L(path_R), 'o-', ms=4, color='#f43f5e', alpha=0.8,
         label=f'출발 w=3.2 → 도착 {path_R[-1]:.2f} (오른쪽 골짜기)')
plt.plot(path_C, L(path_C), 'o-', ms=4, color='#10b981', alpha=0.8,
         label=f'출발 w=-0.6 → 도착 {path_C[-1]:.2f} (왼쪽 골짜기)')
plt.xlabel('가중치 w'); plt.ylabel('손실 L(w)')
plt.title('경사하강 — 시작점에 따라 다른 골짜기(지역 최소값)에 도착')
plt.legend(fontsize=9)
plt.show()

print(f'왼쪽 골짜기 손실 {L(path_C[-1]):.3f}  <  오른쪽 골짜기 손실 {L(path_R[-1]):.3f}')
print('-> 오른쪽 출발은 "덜 좋은" 골짜기에 갇혔다: 지역 최소값 문제!')

## 2. 학습률 실험 — 느림 / 적당 / 발산

같은 출발점(w=3.2)에서 학습률만 바꿔 **걸음 수에 따른 손실**을 비교합니다.

In [ ]:
plt.figure(figsize=(7, 4))
for lr, color, note in [(0.02, '#94a3b8', '너무 작음 (느림)'),
                        (0.15, '#10b981', '적당'),
                        (0.75, '#ef4444', '너무 큼 (발산)')]:
    path = gradient_descent(w0=3.2, lr=lr, n_steps=40)
    losses = L(path)
    plt.plot(losses, 'o-', ms=3, color=color, label=f'η = {lr} — {note}')
    print(f'η = {lr:4}: 40걸음 후 손실 = {losses[-1]:.3f}' + ('  💥 발산!' if losses[-1] > 50 else ''))

plt.ylim(0, 12)
plt.xlabel('걸음 수'); plt.ylabel('손실 L(w)')
plt.title('학습률에 따른 학습 곡선 — AI 에서 가장 자주 만지는 다이얼')
plt.legend()
plt.show()

## 3. 수치미분 — 기울기 공식 검산하기

미분(2부 예고)이 없어도 기울기를 **근사**할 수 있어요. 정의 그대로 —
"아주 조금(h) 움직였을 때 손실이 얼마나 변하나":

$$L'(w) \approx \frac{L(w+h) - L(w-h)}{2h} \quad (h = 0.0001)$$

위에서 쓴 기울기 공식 `dL` 이 맞는지 이 방법으로 검산해 봅니다.
(실무에서도 역전파 구현을 이렇게 검증해요 — gradient check 라고 부릅니다.)

In [ ]:
def numerical_grad(f, w, h=1e-4):
    return (f(w + h) - f(w - h)) / (2 * h)

print(' w     공식 L\'(w)   수치미분    차이')
for w in [-3, -1, 0, 1.5, 3]:
    exact = dL(w)
    approx = numerical_grad(L, w)
    print(f'{w:4} {exact:11.4f} {approx:11.4f} {abs(exact-approx):10.2e}')

print('\n차이가 사실상 0 — 기울기 공식이 정확했습니다. (검산 통과 ✅)')

## 4. 벡터 버전 — 그래디언트 ∇L 로 2차원 골짜기 내려가기

가중치가 2개면 기울기도 2개 — 묶으면 **그래디언트 벡터**입니다.

$$L(w_1, w_2) = w_1^2 + 3w_2^2, \qquad \nabla L = \begin{bmatrix} 2w_1 \\ 6w_2 \end{bmatrix}$$

갱신식은 완전히 같은 모양: $\vec{w} \leftarrow \vec{w} - \eta\,\nabla L$.
숫자가 벡터로 바뀌었을 뿐 — 가중치 수억 개짜리 GPT 도 원리는 이 한 줄입니다.

In [ ]:
def L2(w):
    return w[0] ** 2 + 3 * w[1] ** 2

def grad2(w):
    return np.array([2 * w[0], 6 * w[1]])

w = np.array([4.0, 2.5])     # 시작점
eta = 0.12
path = [w.copy()]
for _ in range(25):
    w = w - eta * grad2(w)   # 벡터 경사하강 — 식은 1변수와 똑같다!
    path.append(w.copy())
path = np.array(path)

# 등고선 + 내려간 경로
g1, g2 = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-3.5, 3.5, 100))
plt.figure(figsize=(7, 4.5))
plt.contour(g1, g2, g1**2 + 3 * g2**2, levels=14, cmap='Blues_r', linewidths=0.8)
plt.plot(path[:, 0], path[:, 1], 'o-', ms=4, color='#f43f5e', label='경사하강 경로')
plt.plot(0, 0, '*', ms=15, color='#10b981', label='최소점 (0,0)')
plt.xlabel('$w_1$'); plt.ylabel('$w_2$')
plt.gca().set_aspect('equal')
plt.title('그래디언트 반대 방향으로 — 등고선을 가로질러 골짜기 바닥으로')
plt.legend()
plt.show()

print('도착점:', path[-1].round(4), '/ 손실:', round(L2(path[-1]), 6))

경로가 등고선과 **수직으로**(가장 가파른 내리막으로) 가로지르는 것도 보이죠 —
그래디언트가 등고선에 수직이기 때문입니다(ch10 직교의 마지막 출연).

---

### 정리

| 개념 | 한 줄 요약 | 코드 |
|------|-----------|------|
| 손실 L(w) | 틀린 정도를 숫자 하나로 | `L(w)` |
| 기울기 | 발밑 경사 (양수=오르막) | `dL(w)`, 검산은 수치미분 |
| 경사하강 | 기울기 반대로 한 걸음씩 | `w -= lr * dL(w)` |
| 학습률 η | 걸음 크기 (작으면 느림, 크면 발산) | `lr` |
| 그래디언트 ∇L | 기울기들의 벡터 — 최급상승 방향 | `w -= lr * grad(w)` |

### 🎓 1부 선형대수 — 전 과정 수료!

**ch1 벡터**에서 시작해 **ch12 역전파**까지. AI 학습 한 사이클(입력→순전파→손실→역전파→갱신)의
수학을 전부 통과했습니다.

다음 여정 — **2부 미적분**: 오늘 블랙박스로 남긴 "기울기를 구해주는 기계(미분)"의 뚜껑을 엽니다.
**3부 확률통계**: AI 의 불확실성과 추론. 1부에서 만든 벡터·행렬 위에서 계속됩니다.